In [1]:
# Parameters
dummy = 0


# Strategy Benchmark Ranking

Tests every implemented strategy on every available ticker, ranks them by return %, and produces a clean leaderboard.

In [2]:
import sys
from pathlib import Path
import pandas as pd
from backtesting import Backtest

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.strategies.sma_crossover import SMACrossoverStrategy
from src.strategies.ema_ribbon import EMARibbonStrategy
from src.strategies.vwap_bounce import VWAPBounceStrategy
from src.strategies.keltner_channel import KeltnerChannelStrategy
from src.strategies.chandelier_exit import ChandelierExitStrategy
from src.strategies.adx_trend_strength import ADXTrendStrengthStrategy
from src.strategies.parabolic_sar import ParabolicSARStrategy
from src.strategies.ichimoku_cloud import IchimokuCloudStrategy
from src.strategies.linear_regression_channel import LinearRegressionChannelStrategy
from src.strategies.rsi_divergence import RSIDivergenceStrategy
from src.strategies.stoch_rsi_crossover import StochRSICrossoverStrategy
from src.strategies.cci_strategy import CCIStrategy
from src.strategies.williams_r_reversal import WilliamsRReversalStrategy
from src.strategies.tsi_strategy import TSIStrategy
from src.strategies.ultimate_oscillator import UltimateOscillatorStrategy
from src.strategies.awesome_oscillator import AwesomeOscillatorStrategy
from src.strategies.chaikin_oscillator import ChaikinOscillatorStrategy

DATA_DIR = project_root / "data" / "raw"

STRATEGIES = {
    "SMA_Cross": SMACrossoverStrategy,
    "EMA_Ribbon": EMARibbonStrategy,
    "VWAP_Bounce": VWAPBounceStrategy,
    "Keltner": KeltnerChannelStrategy,
    "Chandelier": ChandelierExitStrategy,
    "ADX": ADXTrendStrengthStrategy,
    "ParabolicSAR": ParabolicSARStrategy,
    "Ichimoku": IchimokuCloudStrategy,
    "LinReg": LinearRegressionChannelStrategy,
    "RSI_Div": RSIDivergenceStrategy,
    "StochRSI": StochRSICrossoverStrategy,
    "CCI": CCIStrategy,
    "WilliamsR": WilliamsRReversalStrategy,
    "TSI": TSIStrategy,
    "UltOsc": UltimateOscillatorStrategy,
    "AO": AwesomeOscillatorStrategy,
    "Chaikin": ChaikinOscillatorStrategy,
}

C:\Dev\projects\investment_trying\.venv\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
def load_data(filepath: Path) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    for col in ["Datetime", "date", "Date", "timestamp"]:
        if col in df.columns:
            df = df.set_index(col)
            df.index = pd.to_datetime(df.index)
            break
    col_map = {c.lower(): c for c in df.columns}
    for std, variants in [("Open", ["open"]), ("High", ["high"]), ("Low", ["low"]),
                           ("Close", ["close"]), ("Volume", ["volume"])]:
        for v in variants:
            if v in col_map and col_map[v] != std:
                df[std] = df[col_map[v]]
    if "Volume" not in df.columns:
        df["Volume"] = 0
    return df[["Open", "High", "Low", "Close", "Volume"]]

In [4]:
files = sorted(DATA_DIR.glob("*.csv"))
print(f"Testing {len(STRATEGIES)} strategies on {len(files)} assets = {len(STRATEGIES) * len(files)} backtests\n")

results = []
total = len(STRATEGIES) * len(files)
count = 0

for f in files:
    try:
        df = load_data(f)
    except Exception as e:
        print(f"SKIP {f.name}: {e}")
        continue
    if len(df) < 200:
        print(f"SKIP {f.name}: only {len(df)} bars")
        continue

    for name, cls in STRATEGIES.items():
        count += 1
        try:
            bt = Backtest(df, cls, cash=1_000_000, commission=0.001, exclusive_orders=True)
            stats = bt.run()
            results.append({
                "Asset": f.name, "Strategy": name,
                "Trades": int(stats.get("# Trades", 0)),
                "WR": float(stats.get("Win Rate [%]", 0)),
                "Sharpe": float(stats.get("Sharpe Ratio", -999)),
                "PF": float(stats.get("Profit Factor", 0)),
                "Return": float(stats.get("Return [%]", 0)),
                "BnH": float(stats.get("Buy & Hold Return [%]", 0)),
                "MaxDD": float(stats.get("Max. Drawdown [%]", 0)),
            })
        except Exception as e:
            print(f"ERROR {f.name} x {name}: {e}")

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Return", ascending=False).reset_index(drop=True)
print(f"Done: {len(df_results)} results")

Testing 17 strategies on 15 assets = 255 backtests



Backtest.run:   0%|          | 0/17270 [00:00<?, ?bar/s]

C:\Users\Joey Chiu\AppData\Local\Temp\ipykernel_63228\2591785270.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/17469 [00:00<?, ?bar/s]

C:\Users\Joey Chiu\AppData\Local\Temp\ipykernel_63228\2591785270.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


...

C:\Users\Joey Chiu\AppData\Local\Temp\ipykernel_63228\2591785270.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/2482 [00:00<?, ?bar/s]

C:\Users\Joey Chiu\AppData\Local\Temp\ipykernel_63228\2591785270.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest.run:   0%|          | 0/2515 [00:00<?, ?bar/s]

Done: 255 results


In [5]:
# Top 10 by Return
df_results.head(10).style.format({
    "WR": "{:.1f}%", "Sharpe": "{:.2f}", "PF": "{:.2f}",
    "Return": "{:.1f}%", "BnH": "{:.1f}%", "MaxDD": "{:.1f}%"
})

,Asset,Strategy,Trades,WR,Sharpe,PF,Return,BnH,MaxDD
0,BTC_USD_daily.csv,Chaikin,130,30.0%,0.53,1.97,581.4%,2330.8%,-74.5%
1,QQQ_daily.csv,SMA_Cross,4,75.0%,0.75,8.85,336.2%,407.4%,-28.6%
2,BTC_USD_daily.csv,ParabolicSAR,79,36.7%,0.34,1.55,192.6%,2330.8%,-61.3%
3,SPY_daily.csv,SMA_Cross,4,50.0%,0.57,4.22,131.7%,238.2%,-33.7%
4,BTC_USD_daily.csv,EMA_Ribbon,42,33.3%,0.28,1.93,127.5%,2330.8%,-63.8%
5,QQQ_daily.csv,Ichimoku,23,52.2%,0.71,3.20,116.6%,398.6%,-18.5%
6,SPY_daily.csv,UltOsc,22,54.5%,0.49,1.88,87.7%,237.1%,-26.2%
7,IAU_daily.csv,Ichimoku,23,47.8%,0.54,2.94,54.2%,117.1%,-12.1%
8,GLD_daily.csv,SMA_Cross,7,14.3%,0.35,2.45,51.7%,115.2%,-26.9%
9,IAU_daily.csv,SMA_Cross,7,28.6%,0.35,2.47,51.1%,118.1%,-26.8%


In [6]:
# Passing combos (Sharpe >= 0.3, PF >= 1.0)
passing = df_results[(df_results["Sharpe"] >= 0.3) & (df_results["PF"] >= 1.0)]
print(f"Passing: {len(passing)} of {len(df_results)} combos\n")
passing.style.format({
    "WR": "{:.1f}%", "Sharpe": "{:.2f}", "PF": "{:.2f}",
    "Return": "{:.1f}%", "BnH": "{:.1f}%", "MaxDD": "{:.1f}%"
})

Passing: 15 of 255 combos



,Asset,Strategy,Trades,WR,Sharpe,PF,Return,BnH,MaxDD
0,BTC_USD_daily.csv,Chaikin,130,30.0%,0.53,1.97,581.4%,2330.8%,-74.5%
1,QQQ_daily.csv,SMA_Cross,4,75.0%,0.75,8.85,336.2%,407.4%,-28.6%
2,BTC_USD_daily.csv,ParabolicSAR,79,36.7%,0.34,1.55,192.6%,2330.8%,-61.3%
3,SPY_daily.csv,SMA_Cross,4,50.0%,0.57,4.22,131.7%,238.2%,-33.7%
5,QQQ_daily.csv,Ichimoku,23,52.2%,0.71,3.20,116.6%,398.6%,-18.5%
6,SPY_daily.csv,UltOsc,22,54.5%,0.49,1.88,87.7%,237.1%,-26.2%
7,IAU_daily.csv,Ichimoku,23,47.8%,0.54,2.94,54.2%,117.1%,-12.1%
8,GLD_daily.csv,SMA_Cross,7,14.3%,0.35,2.45,51.7%,115.2%,-26.9%
9,IAU_daily.csv,SMA_Cross,7,28.6%,0.35,2.47,51.1%,118.1%,-26.8%
10,SPY_daily.csv,Ichimoku,30,46.7%,0.43,2.22,45.3%,228.1%,-18.3%


In [7]:
# Save to CSV
out_path = project_root / "reports" / "benchmark_ranking.csv"
df_results.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")

Saved to: C:\Dev\projects\investment_trying\reports\benchmark_ranking.csv
